In [1]:
import os
from datetime import datetime, timezone
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from io import StringIO

## input files and directories
homedir = r'./'
reachdir = homedir + r'data/flag_wse/'
geojson_path = homedir + r"js/records_points.geojson"


# Metafile: reach ids for which code needs to run
gbm_file_op = pd.read_csv(homedir + 'inputs/GBM_points.csv')
ids = gbm_file_op["reach_id"]
gbm_file_op = gbm_file_op.drop(columns='records')

## Dates for which we want to get the data
start_time = "2023-01-01T00:00:00Z"
# end_time always current UTC time
end_time = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")



# FP portal address and hydrcron URL, and Fields we want to download
FTS_URL = "https://fts.podaac.earthdata.nasa.gov/v1"
HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"
fields = "reach_id,time_str,wse,reach_q,dark_frac,xovr_cal_q,reach_q_b"


# DEF to query hydrodron on reach_ids
def query_hydrocron(query_url, reach_id, start_time, end_time, fields, empty_df):
    """Query Hydrocron for reach-level time series data.

    Parameters
    ----------
    query_url: str - URL to use to query FTS
    reach_id: str - String SWORD reach identifier
    start_time: str - String time to start query
    end_time: str - String time to end query
    fields: list - List of fields to return in query response
    empty_df: pandas.DataFrame that contains empty query results

    Returns
    -------
    pandas.DataFrame that contains query results
    """

    params = {
        "feature": "Reach",
        "feature_id": reach_id,
        "output": "csv",
        "start_time": start_time,
        "end_time": end_time,
        "fields": fields,
        "collection_name": "SWOT_L2_HR_RiverSP_D",
    }
    results = requests.get(query_url, params=params)
    if "results" in results.json().keys():
        results_csv = results.json()["results"]["csv"]
        df = pd.read_csv(StringIO(results_csv))
    else:
        df = empty_df

    return df


## Iterating through the reaches

for reach in ids:
    pathname = reachdir + str(reach) + '.csv' #condition not to download the same files
    if os.path.exists(pathname) == False:
        #Create an empty dataframe for cases where no data is returned for a reach identifier
        empty_df = pd.DataFrame({
            "reach_id": np.int64(reach),
            "time_str": datetime(1900, 1, 1).strftime("%Y-%m-%dT%H:%M:%S"),
            "wse": -999999999999.0,
            "wse_units": "m"
        }, index=[0])
        results = query_hydrocron(HYDROCRON_URL, reach, start_time, end_time, fields, empty_df)

   
        ddf = results.loc[(results["wse"] != -999999999999.0)]

        # # Convert time_str to datetime format
        ddf.time_str = pd.to_datetime(ddf.time_str)
        ddf['Date'] = ddf.time_str
        ddf.set_index('Date', inplace=True)
        ddf1 = ddf.drop(['time_str'], axis=1)
        ddf1.to_csv(pathname) #downloading csv file for reach_id using ids loop

print("DONE! Downloaded latest SWOt RiverSP Products.")


## Reading files from the dorectory and adding flags

recordgbmdf = pd.DataFrame(columns=["reach_id", "records"])
for reach in ids:
    swot_df = pd.read_csv(reachdir + str(reach) + '.csv')
    if swot_df.shape[0]>1:
        swot_df['Date'] = pd.to_datetime(swot_df['Date']).dt.tz_localize(None)
        good_mask = (swot_df["reach_q"] <= 1) & (swot_df["dark_frac"] < 1) & (swot_df["xovr_cal_q"] < 1) & (swot_df["reach_q_b"] <= 131072)
        bad_mask = (swot_df["reach_q"] >= 3) & (swot_df["dark_frac"] == 1) & (swot_df["xovr_cal_q"] >= 2) & (swot_df["reach_q_b"] > 2097152)
        swot_df["flag"] = 1        # default
        swot_df.loc[bad_mask, "flag"] = 2
        swot_df.loc[good_mask, "flag"] = 0
        swot_df[["Date", "wse", "flag"]].to_csv(reachdir + str(reach)+ "_flag.csv", index=False)
    recordgbmdf.loc[len(recordgbmdf)] = [reach, swot_df.shape[0]]

print("DONE! Flagging applied to the downloaded data.")

# ## Creating the reacords gbm where number of records in each reaches will be saved
merged_gdfgbm_op = gbm_file_op.merge(recordgbmdf, on="reach_id", how="left")

merged_gdfgbm_op = gpd.GeoDataFrame(
    merged_gdfgbm_op,
    geometry=gpd.points_from_xy(
        merged_gdfgbm_op["x"],
        merged_gdfgbm_op["y"]
    ),
    crs="EPSG:4326"
)

geojson_records = merged_gdfgbm_op.to_json()

new_data = "var swotstations = " + geojson_records

with open(geojson_path, "w") as f:
    f.write(new_data)

print("DONE! Modified file saved.", geojson_path)


DONE! Downloaded latest SWOt RiverSP Products.
DONE! Flagging applied to the downloaded data.
DONE! Modified file saved. ./js/records_points.geojson


# BWDB Data post-process to get daily reference WL

In [6]:
import pandas as pd
## Converting BWDB Data into reaadable format
# Read Excel
df = pd.read_excel(r"/Users/nbiswas/Documents/SWOT_Bangladesh/3 Hourly Water Level Data.xlsx")

# Parse datetime
df["Date Time"] = pd.to_datetime(
    df["Date Time"], format="%d/%b/%Y %I:%M:%S %p"
)

# Create date-only column
df["Date Time"] = df["Date Time"].dt.date

# Keep required columns
df = df[["Station ID", "Date Time", "WL (mMSL)"]]

# Daily mean (no missing dates created)
daily_df = (
    df.groupby(["Station ID", "Date Time"], as_index=False)
      .mean()
)

daily_df["WL (mMSL)"] = daily_df["WL (mMSL)"].round(2)

daily_df.to_csv("./data/bwdb_measured_daily_updated.csv", index=False)



FileNotFoundError: [Errno 2] No such file or directory: '/Users/nbiswas/Documents/SWOT_Bangladesh/3 Hourly Water Level Data.xlsx'